# 04 VIST-NRF Method & Ablations — Stage C
State encoder + policy encoder + prediction head, with optional adversarial environment confusion and invariance penalties (ERM/IRM/V-REx/GroupDRO).

In [ ]:
import numpy as np
from vist_nrf.utils.seed import set_seed
from vist_nrf.models import build_model
from vist_nrf.eval.metrics import evaluate

set_seed(0)
rng = np.random.default_rng(1)
X = rng.normal(0, 1, (150, 10))
y = (X[:, 0] > 0).astype(int)
envs = [0] * 75 + [1] * 75

for cfg in [
    dict(use_policy=False, use_adversarial=False, invariance_method='erm'),
    dict(use_policy=True, use_adversarial=False, invariance_method='erm'),
    dict(use_policy=False, use_adversarial=True, invariance_method='erm'),
]:
    m = build_model('vist_nrf', input_dim=10, n_envs=2, epochs=3, **cfg)
    m.fit(X[:100], y[:100], envs=envs[:100])
    p = m.predict_proba(X[100:])[:, 1]
    r = evaluate(y[100:], p)
    print(cfg, '| AUROC=%.3f' % r.auroc)